In [68]:

import mediapipe as mp
import cv2
import os

In [69]:
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

model_path = "hand_landmarker.task"


In [70]:
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode



In [71]:
options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.3
)




In [72]:
# import time
# cap = cv2.VideoCapture(1)
# time.sleep(2)
#
#
# with HandLandmarker.create_from_options(options) as landmarker:
#     cap = cv2.VideoCapture(1)
#
#     if cap.isOpened() is False:
#         print("Error opening video stream or file")
#
#     while cap.isOpened():
#         ret, frame = cap.read()
#         if not ret:
#             print("Could not read frame")
#             break
#
#         h, w = frame.shape[:2]
#         rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
#         result = landmarker.detect(mp_image)
#         if result.hand_landmarks:
#             for hand_landmark in result.hand_landmarks:
#                 for lm in hand_landmark:
#                     x = int(lm.x*w)
#                     y = int(lm.y*h)
#
#                     cv2.circle(frame, (x,y), 3, (0,255,), -1)
#
#
#         cv2.imshow("Frame", frame)
#         if cv2.waitKey(1) == ord('q'):
#             break
#
#     cap.release()
#     cv2.destroyAllWindows()
#





In [73]:
import pandas as pd

handPositions = []
PATH = "/Volumes/T7/videos/Surdobot_VideoBase/07_11/resized_cropped_09_12_1_01_02-1-of-20.mp4"
out_path = "/Volumes/T7/videos/test"
file_name = "resized_cropped_09_12_1_01_02-1-of-20.mp4"
frame_idx = 0
with HandLandmarker.create_from_options(options) as landmarker:
    cap = cv2.VideoCapture(PATH)
    if not cap.isOpened():
        print("Could not open .mp4")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Total frames in video: {total_frames}")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("Could not read frame")
            break

        if frame_idx %3 == 0:
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            res = landmarker.detect(mp_image)
            if res.hand_landmarks:
                h, w = frame.shape[:2]
                for hand_idx, hand_landmarks in enumerate(res.hand_landmarks):
                    wrist = hand_landmarks[0]
                    hand_label = res.handedness[hand_idx][0].display_name

                    row = {
                        "filename": file_name,
                        "frame": frame_idx,
                        "hand_label": hand_label,
                        "wrist_x": wrist.x,
                        "wrist_y": wrist.y
                    }

                    for j, lm in enumerate(hand_landmarks):
                        row[f"x{j}"] = lm.x - wrist.x
                        row[f"y{j}"] = lm.y - wrist.y

                         # draw each landmark
                        cx, cy = int(lm.x * w), int(lm.y * h)
                        cv2.circle(frame, (cx, cy), 2, (0, 255, 0), -1)

                    # draw wrist label
                    wx, wy = int(wrist.x * w), int(wrist.y * h)
                    cv2.putText(frame, hand_label, (wx, wy - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                    handPositions.append(row)

        frame_idx += 1
        cv2.imwrite(f"{out_path}/frame_{frame_idx}.jpg", frame)

df = pd.DataFrame(handPositions)









I0000 00:00:1781005544.927043 1226826 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4
W0000 00:00:1781005544.930531 1226829 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781005544.939835 1226829 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Total frames in video: 652
Could not read frame


In [74]:
df

,filename,frame,hand_label,wrist_x,wrist_y,x0,y0,x1,y1,x2,...,x16,y16,x17,y17,x18,y18,x19,y19,x20,y20
0,resized_cropped_09_12_1_01_02-1-of-20.mp4,0,Left,0.693393,0.623217,0.0,0.0,-0.028577,-0.017075,-0.044230,...,0.050074,-0.134037,0.033418,-0.052362,0.043199,-0.074575,0.049015,-0.090087,0.054979,-0.103036
1,resized_cropped_09_12_1_01_02-1-of-20.mp4,6,Left,0.569273,0.648216,0.0,0.0,-0.032641,-0.016515,-0.057649,...,-0.047640,-0.155563,-0.035383,-0.072796,-0.033531,-0.101418,-0.035613,-0.122563,-0.037591,-0.141057
2,resized_cropped_09_12_1_01_02-1-of-20.mp4,6,Left,0.518726,0.541188,0.0,0.0,-0.035175,0.023246,-0.062142,...,0.055907,0.113903,0.007090,0.022125,0.031533,0.058927,0.048931,0.081760,0.062228,0.097855
3,resized_cropped_09_12_1_01_02-1-of-20.mp4,9,Left,0.604875,0.629152,0.0,0.0,-0.043046,-0.007191,-0.063627,...,-0.069974,-0.153757,-0.020490,-0.102638,-0.035318,-0.135526,-0.046795,-0.148884,-0.053874,-0.155301
4,resized_cropped_09_12_1_01_02-1-of-20.mp4,12,Left,0.597019,0.624282,0.0,0.0,-0.041525,-0.007804,-0.071607,...,-0.073295,-0.153786,-0.032529,-0.083630,-0.046989,-0.109647,-0.056406,-0.126085,-0.063488,-0.140222
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
371,resized_cropped_09_12_1_01_02-1-of-20.mp4,639,Left,0.697303,0.968656,0.0,0.0,-0.048482,0.009334,-0.092789,...,-0.193764,0.050659,-0.102059,0.014992,-0.135295,0.021713,-0.155513,0.027582,-0.172914,0.033910
372,resized_cropped_09_12_1_01_02-1-of-20.mp4,642,Right,0.299170,0.697268,0.0,0.0,0.055240,-0.018601,0.093778,...,0.084258,-0.015379,0.007172,-0.047796,0.039567,-0.030584,0.050131,-0.017152,0.051015,-0.009489
373,resized_cropped_09_12_1_01_02-1-of-20.mp4,642,Right,0.697472,0.969507,0.0,0.0,-0.047971,0.008279,-0.093472,...,-0.193539,0.050923,-0.102708,0.014923,-0.136929,0.022272,-0.156761,0.028284,-0.173450,0.034709
374,resized_cropped_09_12_1_01_02-1-of-20.mp4,645,Right,0.286851,0.670198,0.0,0.0,0.030002,-0.027503,0.068134,...,0.116554,0.000214,0.032947,0.000015,0.069230,0.005206,0.085133,0.010632,0.095789,0.015033


In [75]:
outpath = "/Volumes/T7/videos"
file_name = "frame_24.png"
cap = cv2.VideoCapture(PATH)
frame_id = 0
if not cap.isOpened():
    print("Could not open .mp4")
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Could not read frame")
        break

    h, w = frame.shape[:2]
    if frame_id == 24:

        for i in range(0, 21):
            actual_x = df["wrist_x"].iloc[0] + df[f"x{i}"].iloc[0]
            actual_y = df["wrist_y"].iloc[0] + df[f"y{i}"].iloc[0]

            x, y = int(actual_x*w), int(actual_y*h)
            cv2.circle(frame, (x, y), 3, (0, 255, 0), -1)

        cv2.imwrite(os.path.join(outpath, file_name), frame)
        break
    if cv2.waitKey(1) == ord('q'):
        break
    frame_id += 1

cap.release()
cv2.destroyAllWindows()



In [76]:
df.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['filename', 'frame', 'hand_label', 'wrist_x', 'wrist_y', 'x0', 'y0',
       'x1', 'y1', 'x2', 'y2', 'x3', 'y3', 'x4', 'y4', 'x5', 'y5', 'x6', 'y6',
       'x7', 'y7', 'x8', 'y8', 'x9', 'y9', 'x10', 'y10', 'x11', 'y11', 'x12',
       'y12', 'x13', 'y13', 'x14', 'y14', 'x15', 'y15', 'x16', 'y16', 'x17',
       'y17', 'x18', 'y18', 'x19', 'y19', 'x20', 'y20'],
      dtype='str')>

In [77]:
print(df["frame"].iloc[0])      # what frame is this row from?
print(df["filename"].iloc[0])   # what video is this row from?
print(df["hand_label"].iloc[0]) # Left or Right?
print(df["wrist_x"].iloc[0], df["wrist_y"].iloc[0])  # where is the wrist?

0
resized_cropped_09_12_1_01_02-1-of-20.mp4
Left
0.6933934688568115 0.6232165098190308


In [78]:
# Check frames where 2 hands were detected
two_hand_frames = df.groupby("frame").filter(lambda x: len(x) > 1)
print(two_hand_frames[["frame", "hand_label", "wrist_x", "wrist_y"]].head(10))

# Count label distribution

    frame hand_label   wrist_x   wrist_y
1       6       Left  0.569273  0.648216
2       6       Left  0.518726  0.541188
6      21      Right  0.305510  0.700251
7      21       Left  0.721354  0.710269
8      24      Right  0.282540  0.679874
9      24       Left  0.759510  0.652439
10     27       Left  0.763528  0.660986
11     27      Right  0.266951  0.697657
12     30       Left  0.767305  0.697087
13     30      Right  0.288126  0.746758


In [81]:
df.to_csv("hand_labels.csv", index=False)